<a href="https://colab.research.google.com/github/AmarnathSharma10/recommendation-systems/blob/main/pratlip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
chapters=pd.read_csv("/content/chapters.csv")
interactions=pd.read_csv("/content/interactions.csv")

In [6]:
chapters.head()

,chapter_id,chapter_sequence_no,book_id,author_id,published_date,tags
0,2812946,1,139726,66847,1990-03-22,Fantasy|Horror
1,4330764,2,139726,66847,1990-04-09,Fantasy|Young Adult|Literary Fiction
2,2664499,3,139726,66847,1990-04-07,Fantasy
3,2260666,4,139726,66847,1990-05-18,Literary Fiction|Fantasy
4,6069976,1,191772,62262,2008-07-30,Horror|Young Adult|Romance|Graphic Novel


In [7]:
interactions.head()

,user_id,chapter_id,book_id
0,user_2378720,5894067,444295
1,user_2321122,2532511,785684
2,user_2335775,6777764,999595
3,user_7906001,7366896,748410
4,user_9981689,7853186,418083


In [10]:
df = interactions.merge(chapters, on=["chapter_id", "book_id"])

In [11]:
user_book = df.groupby(['user_id', 'book_id'])['chapter_sequence_no'].max().reset_index()
total_chapters = chapters.groupby('book_id')['chapter_sequence_no'].max().reset_index()
total_chapters.rename(columns={'chapter_sequence_no': 'total_chapters'}, inplace=True)
user_book = user_book.merge(total_chapters, on='book_id')

user_book['progress'] = user_book['chapter_sequence_no'] / user_book['total_chapters']
user_book['weight'] = user_book['progress'] ** 2

In [12]:
book_meta = chapters.groupby('book_id').agg({
    'tags': 'first',
    'author_id': 'first',
    'published_date': 'first'
}).reset_index()

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tag_matrix = tfidf.fit_transform(book_meta['tags'].fillna(""))

In [14]:
author_matrix = pd.get_dummies(book_meta['author_id'])
book_meta['year'] = pd.to_datetime(book_meta['published_date']).dt.year

year = book_meta['year']
year_norm = (year - year.min()) / (year.max() - year.min())

In [19]:
from scipy.sparse import hstack

book_features = hstack([
    tag_matrix,
    author_matrix.values,
    year_norm.values.reshape(-1, 1)
])

book_features = book_features.tocsr()

In [20]:
book_id_to_index = {b: i for i, b in enumerate(book_meta['book_id'])}
index_to_book_id = {i: b for b, i in book_id_to_index.items()}

In [21]:
user_vectors = {}

for user, group in user_book.groupby('user_id'):
    vec = None
    total_w = 0

    for _, row in group.iterrows():
        b = row['book_id']
        if b not in book_id_to_index:
            continue

        idx = book_id_to_index[b]
        w = row['weight']

        if vec is None:
            vec = w * book_features[idx]
        else:
            vec += w * book_features[idx]

        total_w += w

    if vec is not None and total_w > 0:
        vec = vec / total_w

    user_vectors[user] = vec

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(user_id, top_k=10):
    user_vec = user_vectors[user_id]

    scores = cosine_similarity(user_vec, book_features).flatten()

    seen_books = set(user_book[user_book.user_id == user_id]['book_id'])

    ranked = [
        (index_to_book_id[i], score)
        for i, score in enumerate(scores)
        if index_to_book_id[i] not in seen_books
    ]

    ranked.sort(key=lambda x: x[1], reverse=True)

    return [b for b, _ in ranked[:top_k]]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def evaluate(user_sequences, k=10):
    hits = 0
    mrr = 0
    total = 0

    for user, books in user_sequences.items():
        if len(books) < 2:
            continue

        train_books = books[:-1]
        test_book = books[-1]

        # build user vector from train_books only
        vec = None
        total_w = 0

        for b in train_books:
            if b not in book_id_to_index:
                continue

            idx = book_id_to_index[b]

            # fetch weight
            w = user_book[(user_book.user_id == user) & (user_book.book_id == b)]['weight'].values
            if len(w) == 0:
                continue
            w = w[0]

            if vec is None:
                vec = w * book_features[idx]
            else:
                vec += w * book_features[idx]

            total_w += w

        if vec is None or total_w == 0:
            continue

        vec = vec / total_w

        # compute scores
        scores = cosine_similarity(vec, book_features).flatten()

        # remove seen books
        seen = set(train_books)

        ranked = [
            (index_to_book_id[i], s)
            for i, s in enumerate(scores)
            if index_to_book_id[i] not in seen
        ]

        ranked.sort(key=lambda x: x[1], reverse=True)

        top_k = [b for b, _ in ranked[:k]]

        # HR@K
        if test_book in top_k:
            hits += 1

            # MRR
            rank = top_k.index(test_book) + 1
            mrr += 1 / rank

        total += 1

    hr = hits / total if total else 0
    mrr = mrr / total if total else 0

    return hr, mrr

user_book_sorted = user_book.sort_values(
    ['user_id', 'chapter_sequence_no'], ascending=[True, True]
)

user_sequences = user_book_sorted.groupby('user_id')['book_id'].apply(list).to_dict()
hr, mrr = evaluate(user_sequences, k=10)

print("HR@10:", hr)
print("MRR:", mrr)

# problem staement:
## Given a user's history of books they've read (inferred from chapter interactions), predict which new book they are most likely to read next — ranking a set of candidate books by relevance to that user.

In [2]:
!pip install implicit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
"""
Next Chapter Recommendation System
=====================================

PROBLEM
-------
A user is reading multiple books at once.
Predict which book they will pick up next, then return the next chapter in that book.

APPROACH
--------
No timestamps exist in the data, so recency is unknowable.
Instead we use two signals that ARE in the data:

  progress          = max chapter reached / total chapters in book
                      (how far they got — the strongest engagement signal)

  interaction_count = number of times they touched that book
                      (how often they came back to it)

Final score per (user, book):
  score = 0.7 * progress + 0.3 * (count / user's own max count)

The book with the highest score is predicted as "next to continue".
Next chapter = max chapter they reached + 1 (if it exists in the dataset).

EVALUATION
----------
No timestamps → we simulate ground truth with leave-one-out:
  - For each user, hold out their highest-scoring book as "ground truth"
  - Predict from remaining books
  - Check if prediction matches held-out book

Metrics:
  HR@1  — did we predict the right book at rank 1?
  MRR   — mean reciprocal rank (rewards being close even when not exactly right)

HOW TO RUN
----------
  pip install pandas numpy
  python recommender.py
  (chapters.csv and interactions.csv must be in the same folder)
"""

import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
CHAPTERS_PATH     = "chapters.csv"
INTERACTIONS_PATH = "interactions.csv"
PROGRESS_WEIGHT   = 0.7   # weight for progress in scoring formula
COUNT_WEIGHT      = 0.3   # weight for interaction count (must sum to 1)
MIN_BOOKS         = 2     # minimum books per user (need at least 2 for eval)
EVAL_SAMPLE       = 5000  # number of users to evaluate (None = all)
RANDOM_SEED       = 42

np.random.seed(RANDOM_SEED)


# ── Step 1: Load data ─────────────────────────────────────────────────────────
def load_data():
    print("=" * 55)
    print("  Next Chapter Recommendation System")
    print("=" * 55)
    print("\n[1/5] Loading data...")

    chapters     = pd.read_csv(CHAPTERS_PATH)
    interactions = pd.read_csv(INTERACTIONS_PATH)

    # Merge so each interaction row has chapter_sequence_no
    df = interactions.merge(
        chapters[["chapter_id", "book_id", "chapter_sequence_no"]],
        on=["chapter_id", "book_id"],
        how="left"
    )

    print(f"      interactions : {len(df):>10,}")
    print(f"      unique users : {df['user_id'].nunique():>10,}")
    print(f"      unique books : {df['book_id'].nunique():>10,}")

    return df, chapters


# ── Step 2: Build user-book summary ──────────────────────────────────────────
def build_user_book_summary(df, chapters):
    """
    For every (user, book) pair compute:
      chapter_sequence_no  — max chapter the user reached in that book
      total_chapters       — how many chapters the book has in total
      progress             — max_chapter / total_chapters  (0.0 to 1.0)
      count                — how many times the user interacted with that book
      count_norm           — count normalised within each user (0.0 to 1.0)
      score                — final score used for ranking
    """
    print("\n[2/5] Building user-book summary...")

    # (a) max chapter reached per (user, book)
    user_book = (
        df.groupby(["user_id", "book_id"])["chapter_sequence_no"]
        .max()
        .reset_index()
    )

    # (b) total chapters per book
    total_chapters = (
        chapters.groupby("book_id")["chapter_sequence_no"]
        .max()
        .reset_index()
        .rename(columns={"chapter_sequence_no": "total_chapters"})
    )

    # (c) merge and compute progress
    user_book = user_book.merge(total_chapters, on="book_id")
    user_book["progress"] = (
        user_book["chapter_sequence_no"] / user_book["total_chapters"]
    ).clip(0, 1)

    # (d) interaction count per (user, book)
    interaction_count = (
        df.groupby(["user_id", "book_id"])
        .size()
        .reset_index(name="count")
    )
    user_book = user_book.merge(interaction_count, on=["user_id", "book_id"])

    # (e) normalise count WITHIN each user so heavy readers don't dominate
    user_book["count_norm"] = user_book.groupby("user_id")["count"].transform(
        lambda x: x / x.max()
    )

    # (f) final score
    # Mid-progress boost (penalize very low and very high progress)
    user_book["progress_score"] = user_book["progress"] * (1 - user_book["progress"])

    # Final score
    user_book["score"] = (
        user_book["progress_score"]
        + 0.1 * user_book["count_norm"]
    )

    # (g) drop users with fewer than MIN_BOOKS (can't evaluate them)
    books_per_user = user_book.groupby("user_id")["book_id"].nunique()
    valid_users    = books_per_user[books_per_user >= MIN_BOOKS].index
    user_book      = user_book[user_book["user_id"].isin(valid_users)]

    print(f"      users        : {user_book['user_id'].nunique():>10,}")
    print(f"      books        : {user_book['book_id'].nunique():>10,}")
    print(f"      (user, book) pairs : {len(user_book):>7,}")

    return user_book


# ── Step 3: Predict next book for a single user ───────────────────────────────
def predict_next_book(user_id, user_book):
    """
    For a given user, return the book_id with the highest score.
    Optionally pass exclude_books to hide certain books (used during eval).
    """
    books = user_book[user_book["user_id"] == user_id]
    if books.empty:
        return None
    best_row = books.loc[books["score"].idxmax()]
    return best_row["book_id"]


# ── Step 4: Evaluate ─────────────────────────────────────────────────────────
def evaluate(user_book, sample=EVAL_SAMPLE):
    print("\n[3/5] Evaluating (fixed)...")

    all_users = user_book["user_id"].unique()
    if sample and len(all_users) > sample:
        all_users = np.random.choice(all_users, size=sample, replace=False)

    hr1_list = []
    mrr_list = []

    for user_id in all_users:
        u_data = user_book[user_book["user_id"] == user_id].copy()

        if len(u_data) < 2:
            continue

        # ---- Step 1: Random holdout ----
        held_out_row = u_data.sample(1).iloc[0]
        held_out_book = held_out_row["book_id"]

        train = u_data[u_data["book_id"] != held_out_book].copy()

        # ---- Step 2: Recompute ranking on train ----
        # (IMPORTANT: do NOT include held-out in scoring features)
        train = train.sort_values("score", ascending=False)

        # ---- Step 3: Re-rank INCLUDING held-out ----
        # simulate prediction: rank all candidates
        candidates = pd.concat([train, held_out_row.to_frame().T], ignore_index=True)

        candidates = candidates.sort_values("score", ascending=False)

        ranked_list = candidates["book_id"].tolist()

        # ---- Step 4: Metrics ----
        rank = ranked_list.index(held_out_book) + 1

        hr1_list.append(1 if rank == 1 else 0)
        mrr_list.append(1.0 / rank)

    hr1 = np.mean(hr1_list)
    mrr = np.mean(mrr_list)

    avg_books  = user_book.groupby("user_id")["book_id"].nunique().mean()
    random_hr1 = 1.0 / avg_books

    print(f"\n  HR@1 : {hr1:.4f}")
    print(f"  MRR  : {mrr:.4f}")
    print(f"  Random HR@1 : {random_hr1:.4f}")
    print(f"  Lift : {hr1/random_hr1:.2f}x")

    return {"HR@1": hr1, "MRR": mrr}
def demo_predictions(user_book, chapters, n=3):
    print(f"\n[4/5] Sample predictions (fixed)")
    print("=" * 55)

    chapter_lookup = (
        chapters
        .set_index(["book_id", "chapter_sequence_no"])["chapter_id"]
        .to_dict()
    )

    sample_users = np.random.choice(
        user_book["user_id"].unique(), size=n, replace=False
    )

    for user_id in sample_users:
        u_data = user_book[user_book["user_id"] == user_id].copy()

        if len(u_data) < 2:
            continue

        # ---- Holdout ----
        held_out = u_data.sample(1).iloc[0]
        held_out_book = held_out["book_id"]

        train = u_data[u_data["book_id"] != held_out_book].copy()
        train = train.sort_values("score", ascending=False)

        print(f"\nUser : {user_id}")
        print(f"Held-out (ground truth): {held_out_book}")
        print(f"\nTrain books:")

        for _, row in train.iterrows():
            print(f"  book={row['book_id']}  "
                  f"progress={row['progress']:.2f}  "
                  f"count={row['count']}  "
                  f"score={row['score']:.3f}")

        # ---- Prediction ----
        predicted_book = train.iloc[0]["book_id"]

        print(f"\nPredicted next book: {predicted_book}")

        # ---- Check correctness ----
        if predicted_book == held_out_book:
            print("✅ Correct prediction")
        else:
            print("❌ Incorrect prediction")

        # ---- Predict next chapter for predicted book ----
        pred_row = u_data[u_data["book_id"] == predicted_book].iloc[0]
        last_chapter = int(pred_row["chapter_sequence_no"])
        next_seq = last_chapter + 1
        next_chapter = chapter_lookup.get((predicted_book, next_seq))

        print(f"Last chapter read: {last_chapter}")

        if next_chapter:
            print(f"Next chapter: {next_chapter} (#{next_seq})")
        else:
            print("No next chapter found (book may be complete)")

        print("─" * 50)
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    # 1. Load
    df, chapters = load_data()

    # 2. Build summary
    user_book = build_user_book_summary(df, chapters)

    # 3. Evaluate
    results = evaluate(user_book)

    # 4. Demo
    demo_predictions(user_book, chapters)

    print("\n[5/5] Done.")
    return results


if __name__ == "__main__":
    main()

  Next Chapter Recommendation System

[1/5] Loading data...
      interactions :  1,000,000
      unique users :    149,803
      unique books :      9,575

[2/5] Building user-book summary...
      users        :    148,548
      books        :      9,575
      (user, book) pairs : 998,265

[3/5] Evaluating (fixed)...

  HR@1 : 0.1512
  MRR  : 0.3854
  Random HR@1 : 0.1488
  Lift : 1.02x

[4/5] Sample predictions (fixed)

User : user_5338517
Held-out (ground truth): 598597

Train books:
  book=473973  progress=0.30  count=1  score=0.310
  book=161775  progress=1.00  count=1  score=0.100
  book=262934  progress=1.00  count=1  score=0.100

Predicted next book: 473973
❌ Incorrect prediction
Last chapter read: 6
Next chapter: 2273076 (#7)
──────────────────────────────────────────────────

User : user_6539567
Held-out (ground truth): 813944

Train books:
  book=243006  progress=0.80  count=1  score=0.260

Predicted next book: 243006
❌ Incorrect prediction
Last chapter read: 8
Next chapter

In [ ]:
"""
Cold-start recommender for the chapters/interactions dataset.

Strategy is staged by how much history we have for the user:

    Stage 0  (0 interactions):   Diverse popularity via MMR
    Stage 1  (1 interaction):    Item-item collaborative filtering
    Stage 2+ (2-5 interactions): CF blended with content (genre) similarity

Task framing
------------
We recommend BOOKS (not chapters) to a cold user. Ground truth is the
next *distinct* book the user picks up in their interaction log. We
assume the row order in interactions.csv is chronological, since there
is no timestamp column.

Usage
-----
    python coldstart_recsys.py --chapters chapters.csv --interactions interactions.csv

Outputs
-------
    coldstart_results.csv   -- Hit@10 and MRR by model x history length
    coldstart_results.png   -- the same, as two line plots
"""

from __future__ import annotations

import argparse
from collections import defaultdict
from typing import Dict, List, Set

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, lil_matrix

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False


# ============================================================
# Data loading
# ============================================================

def load_data(chapters_path: str, interactions_path: str):
    chapters = pd.read_csv(chapters_path)
    interactions = pd.read_csv(interactions_path)
    # No timestamp column -> we assume row order is chronological.
    interactions = interactions.reset_index(drop=True)
    interactions["event_order"] = interactions.index
    return chapters, interactions


def build_book_metadata(chapters: pd.DataFrame):
    """Aggregate chapter-level metadata to the book level."""
    exploded = (
        chapters.assign(genre=chapters["tags"].fillna("").str.split("|"))
                .explode("genre")
    )
    exploded["genre"] = exploded["genre"].str.strip()
    exploded = exploded[exploded["genre"] != ""]

    book_genres: Dict[int, Set[str]] = (
        exploded.groupby("book_id")["genre"].apply(set).to_dict()
    )

    # First chapter per book (useful if we ever want to return a chapter,
    # not a book, to the caller).
    first_chapters: Dict[int, int] = (
        chapters.sort_values(["book_id", "chapter_sequence_no"])
                .groupby("book_id")["chapter_id"].first().to_dict()
    )
    return book_genres, first_chapters


def build_user_book_history(interactions: pd.DataFrame) -> Dict[str, List[int]]:
    """
    Per user, return the ordered list of DISTINCT books they engaged with,
    in the order they were first touched.
    """
    first_touch = (
        interactions.groupby(["user_id", "book_id"])["event_order"]
                    .min().reset_index()
                    .sort_values(["user_id", "event_order"])
    )
    return first_touch.groupby("user_id")["book_id"].apply(list).to_dict()


def split_users(user_history: Dict[str, List[int]], test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    users = np.array(list(user_history.keys()))
    rng.shuffle(users)
    n_test = int(len(users) * test_frac)
    return set(users[n_test:]), set(users[:n_test])  # train, test


# ============================================================
# Models
# ============================================================

class PopularityRecommender:
    """Baseline: recommend globally most-read books (by unique users)."""

    def fit(self, train_histories):
        counter: Dict[int, int] = defaultdict(int)
        for books in train_histories.values():
            for b in set(books):
                counter[b] += 1
        self.pop = counter
        self.ranked = sorted(counter, key=counter.get, reverse=True)
        return self

    def recommend(self, history, k=10):
        seen = set(history)
        out = []
        for b in self.ranked:
            if b not in seen:
                out.append(b)
                if len(out) == k:
                    break
        return out


class DiversePopularityRecommender:
    """
    Stage 0. Pick a slate that is popular AND spread across genres, so a
    single click gives us a strong preference signal (maximal marginal
    relevance over the top-N popular books).

    Note: on Hit@K this will usually look WORSE than plain popularity. Its
    value is in information gain on click, which this metric does not
    reward directly. We include it for honesty in the comparison.
    """

    def __init__(self, lambda_=0.6, pool_size=500):
        self.lambda_ = lambda_
        self.pool_size = pool_size

    def fit(self, train_histories, book_genres):
        counter: Dict[int, int] = defaultdict(int)
        for books in train_histories.values():
            for b in set(books):
                counter[b] += 1
        self.pop = counter
        self.book_genres = book_genres
        self.pool = sorted(counter, key=counter.get, reverse=True)[: self.pool_size]
        mx = max((counter[b] for b in self.pool), default=1)
        self.pop_score = {b: counter[b] / mx for b in self.pool}
        return self

    @staticmethod
    def _jaccard(a: Set[str], b: Set[str]) -> float:
        if not a or not b:
            return 0.0
        return len(a & b) / len(a | b)

    def recommend(self, history, k=10):
        seen = set(history)
        selected: List[int] = []
        selected_genres: List[Set[str]] = []
        pool = [b for b in self.pool if b not in seen]

        while pool and len(selected) < k:
            best_b, best_s = None, -np.inf
            for b in pool:
                bg = self.book_genres.get(b, set())
                sim = (
                    max(self._jaccard(bg, g) for g in selected_genres)
                    if selected_genres else 0.0
                )
                s = self.lambda_ * self.pop_score[b] - (1 - self.lambda_) * sim
                if s > best_s:
                    best_s, best_b = s, b
            selected.append(best_b)
            selected_genres.append(self.book_genres.get(best_b, set()))
            pool.remove(best_b)
        return selected


class ItemItemCF:
    """
    Stage 1. Cosine similarity over the (user x book) binary matrix.
    For a history, sum the similarity rows of each history book and rank.
    """

    def fit(self, train_histories):
        users = list(train_histories.keys())
        books = sorted({b for bs in train_histories.values() for b in bs})
        self.user_to_idx = {u: i for i, u in enumerate(users)}
        self.book_to_idx = {b: i for i, b in enumerate(books)}
        self.idx_to_book = {i: b for b, i in self.book_to_idx.items()}

        rows, cols = [], []
        for u, bs in train_histories.items():
            ui = self.user_to_idx[u]
            for b in set(bs):
                rows.append(ui)
                cols.append(self.book_to_idx[b])
        data = np.ones(len(rows), dtype=np.float32)
        U = csr_matrix((data, (rows, cols)), shape=(len(users), len(books)))

        # column-normalize U so that U_norm^T @ U_norm gives cosine similarity
        col_norms = np.sqrt(np.asarray(U.multiply(U).sum(axis=0)).ravel())
        col_norms[col_norms == 0] = 1.0
        D = csr_matrix(
            (1.0 / col_norms, (np.arange(len(books)), np.arange(len(books))))
        )
        self.U_norm = U @ D
        self.book_pop = np.asarray(U.sum(axis=0)).ravel()
        return self

    def _similar_to(self, book_idx: int) -> np.ndarray:
        col = self.U_norm[:, book_idx]
        return (self.U_norm.T @ col).toarray().ravel()

    def score(self, history) -> np.ndarray:
        agg = np.zeros(len(self.book_to_idx), dtype=np.float32)
        for b in history:
            if b in self.book_to_idx:
                agg += self._similar_to(self.book_to_idx[b])
        return agg

    def recommend(self, history, k=10):
        seen = set(history)
        if not history:
            # empty history -> fall back to popularity
            order = np.argsort(-self.book_pop)
            return [self.idx_to_book[i] for i in order[:k]]

        agg = self.score(history)
        if agg.max() <= 0:  # no signal, fall back
            order = np.argsort(-self.book_pop)
            out = []
            for i in order:
                b = self.idx_to_book[i]
                if b not in seen:
                    out.append(b)
                    if len(out) == k:
                        break
            return out

        for b in seen:
            if b in self.book_to_idx:
                agg[self.book_to_idx[b]] = -np.inf
        top = np.argpartition(-agg, min(k, len(agg) - 1))[:k]
        top = top[np.argsort(-agg[top])]
        return [self.idx_to_book[i] for i in top]


class BlendedRecommender:
    """
    Stage 2+. CF signal + genre-based content signal. The blend weight
    adapts to history length: with 1 history book, CF is noisy, so we
    lean more on content; with 5+, we trust CF more.
    """

    def fit(self, train_histories, book_genres):
        self.cf = ItemItemCF().fit(train_histories)
        genres = sorted({g for gs in book_genres.values() for g in gs})
        self.genre_to_idx = {g: i for i, g in enumerate(genres)}

        n_books, n_genres = len(self.cf.book_to_idx), len(genres)
        G = lil_matrix((n_books, n_genres), dtype=np.float32)
        for b, bi in self.cf.book_to_idx.items():
            for g in book_genres.get(b, ()):
                gi = self.genre_to_idx.get(g)
                if gi is not None:
                    G[bi, gi] = 1.0
        G = G.tocsr()

        row_norms = np.sqrt(np.asarray(G.multiply(G).sum(axis=1)).ravel())
        row_norms[row_norms == 0] = 1.0
        D = csr_matrix(
            (1.0 / row_norms, (np.arange(n_books), np.arange(n_books)))
        )
        self.G_norm = D @ G
        return self

    def _alpha(self, n_history: int) -> float:
        # grows with history length: 0.50 at 1 book -> 0.85 at 5+ books
        return min(0.85, 0.40 + 0.10 * n_history)

    def recommend(self, history, k=10):
        if not history:
            return self.cf.recommend(history, k)

        cf_score = self.cf.score(history).copy()
        mx = cf_score.max()
        if mx > 0:
            cf_score /= mx

        # user's genre profile = normalized mean of their books' genre vectors
        user_vec = np.zeros(len(self.genre_to_idx), dtype=np.float32)
        n_in = 0
        for b in history:
            bi = self.cf.book_to_idx.get(b)
            if bi is not None:
                user_vec += self.G_norm[bi].toarray().ravel()
                n_in += 1
        if n_in:
            user_vec /= n_in
        nv = np.linalg.norm(user_vec)
        if nv > 0:
            user_vec /= nv
        content_score = self.G_norm @ user_vec

        alpha = self._alpha(len(history))
        final = alpha * cf_score + (1 - alpha) * content_score

        seen = set(history)
        for b in seen:
            bi = self.cf.book_to_idx.get(b)
            if bi is not None:
                final[bi] = -np.inf

        top = np.argpartition(-final, min(k, len(final) - 1))[:k]
        top = top[np.argsort(-final[top])]
        return [self.cf.idx_to_book[i] for i in top]


# ============================================================
# Evaluation
# ============================================================

def evaluate(recommender, test_histories, history_length, k=10):
    """
    For each test user with at least (history_length + 1) distinct books,
    feed the first `history_length` books, and check whether book
    (history_length + 1) appears in the top-k recommendations.
    """
    hits, rrs = [], []
    for _, books in test_histories.items():
        if len(books) <= history_length:
            continue
        hist = books[:history_length]
        target = books[history_length]
        recs = recommender.recommend(hist, k=k)
        if target in recs:
            hits.append(1)
            rrs.append(1.0 / (recs.index(target) + 1))
        else:
            hits.append(0)
            rrs.append(0.0)
    return (
        float(np.mean(hits)) if hits else 0.0,
        float(np.mean(rrs)) if rrs else 0.0,
        len(hits),
    )


def run_grid(models, test_histories, history_lengths, k=10):
    rows = []
    for name, model in models.items():
        for n in history_lengths:
            hit, mrr, n_users = evaluate(model, test_histories, n, k=k)
            rows.append({
                "model": name,
                "history_length": n,
                f"Hit@{k}": round(hit, 4),
                "MRR": round(mrr, 4),
                "n_eval_users": n_users,
            })
    return pd.DataFrame(rows)


# ============================================================
# Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--chapters", default="chapters.csv")
    parser.add_argument("--interactions", default="interactions.csv")
    parser.add_argument("--k", type=int, default=10)
    parser.add_argument(
        "--history-lengths", type=int, nargs="+", default=[0, 1, 2, 3, 5]
    )
    args = parser.parse_args()

    print("Loading data...")
    chapters, interactions = load_data(args.chapters, args.interactions)
    print(f"  {len(chapters):,} chapters, {len(interactions):,} interactions")

    print("Building book metadata and user histories...")
    book_genres, _first_chapters = build_book_metadata(chapters)
    user_hist = build_user_book_history(interactions)
    print(f"  {len(book_genres):,} books, {len(user_hist):,} users")

    train_users, test_users = split_users(user_hist)
    train_hist = {u: user_hist[u] for u in train_users}
    test_hist = {u: user_hist[u] for u in test_users}
    print(f"  {len(train_hist):,} train users / {len(test_hist):,} test users")

    print("Fitting models...")
    pop = PopularityRecommender().fit(train_hist)
    divpop = DiversePopularityRecommender(lambda_=0.6).fit(train_hist, book_genres)
    cf = ItemItemCF().fit(train_hist)
    blend = BlendedRecommender().fit(train_hist, book_genres)

    models = {
        "popularity": pop,
        "diverse_popularity": divpop,
        "item_item_cf": cf,
        "blended_cf+content": blend,
    }

    print(f"Evaluating at history lengths {args.history_lengths}, k={args.k}...")
    results = run_grid(models, test_hist, args.history_lengths, k=args.k)
    print("\n" + results.to_string(index=False))
    results.to_csv("coldstart_results.csv", index=False)

    if HAS_PLT:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        for name, g in results.groupby("model"):
            ax1.plot(g["history_length"], g[f"Hit@{args.k}"], marker="o", label=name)
            ax2.plot(g["history_length"], g["MRR"], marker="o", label=name)
        ax1.set_xlabel("History length (# books)")
        ax1.set_ylabel(f"Hit@{args.k}")
        ax1.set_title(f"Hit@{args.k} vs history length")
        ax1.legend()
        ax2.set_xlabel("History length (# books)")
        ax2.set_ylabel("MRR")
        ax2.set_title("MRR vs history length")
        ax2.legend()
        plt.tight_layout()
        plt.savefig("coldstart_results.png", dpi=120)
        print("\nSaved coldstart_results.csv and coldstart_results.png")
    else:
        print("\nSaved coldstart_results.csv (matplotlib not available, no plot)")


if __name__ == "__main__":
    main()